In [1]:
!pip install pandas langdetect

In [2]:
#import libraries
from pathlib import Path
import json
import re
import html
import pandas as pd
import nltk
from nltk.corpus import stopwords
from langdetect import detect, LangDetectException

In [3]:
#define folder paths
raw_folder = Path("../data/raw")
processed_folder = Path("../data/processed")

#create a empty lists to store comment for Q3 and Q4
records_q3 = []
records_q4 = []

#create an empty lists to store video information for Q3 and Q4
videos_q3 = []
videos_q4 = []

#reads each raw JSON file in the raw data folder
for path in raw_folder.glob("*.json"):
    file_name = path.name
    stem = path.stem

    #seperates the files based on file name: Q3 and Q4
    if "_Q3" in stem:
        sub_question = "Q3"
    elif "_Q4" in stem:
        sub_question = "Q4"
    else:
        print(f"Skipped unknown file: {file_name}")
        continue

    #reconstruct search query from filename
    search_query = (
        stem.replace("_Q3", "")
            .replace("_Q4", "")
            .replace("_", " ")
    )
    
    #loads the JSON files
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    #extracts video information and comment data
    for video in data.get("videos", []):
        video_id = video.get("videoId", "")
        video_title = html.unescape(video.get("title", ""))
        channel = video.get("channelTitle", "")
        published_at = video.get("publishedAt", "")
        view_count = video.get("viewCount", 0)
        like_count = video.get("likeCount", 0)
        #collects the comments 
        comments = video.get("comments", [])
        
        #creates a row of video information data
        video_row = {
            "source_file": file_name,
            "sub_question": sub_question,
            "search_query": search_query,
            "video_id": video_id,
            "video_title": video_title,
            "channel": channel,
            "published_at": published_at,
            "view_count": view_count,
            "like_count": like_count,
            "comment_count_collected": len(comments),
            "video_url": f"https://www.youtube.com/watch?v={video_id}"
        }
        
        #stores the video row in the correct list depending on whether it is Q3 or Q4
        if sub_question == "Q3":
            videos_q3.append(video_row)
        else:
            videos_q4.append(video_row)
        
        #loops through every comment of each video
        for comment in comments:
            #extract the comment texts
            text = html.unescape(comment.get("text", "") or "")
            text = text.replace("\n", " ").replace("\r", " ").strip()
            
            #create one row of comment data
            comment_row = {
                "source_file": file_name,
                "sub_question": sub_question,
                "search_query": search_query,
                "video_id": video_id,
                "video_title": video_title,
                "channel": channel,
                "video_published_at": published_at,
                "video_view_count": view_count,
                "video_like_count": like_count,
                "comment_author": comment.get("author", ""),
                "comment_text": text,
                "comment_published_at": comment.get("publishedAt", ""),
                "comment_like_count": comment.get("likeCount", 0),
                "video_url": f"https://www.youtube.com/watch?v={video_id}"
            }
            
            #store the comment row in the correct list depending if it's Q3 or Q4
            if sub_question == "Q3":
                records_q3.append(comment_row)
            else:
                records_q4.append(comment_row)

#convert the comments into pandas DataFrames
comments_q3 = pd.DataFrame(records_q3)
comments_q4 = pd.DataFrame(records_q4)

#convert the video information into pandas DataFrames
videos_q3_df = pd.DataFrame(videos_q3)
videos_q4_df = pd.DataFrame(videos_q4)

#print the number of comments for Q3 and Q4
print("Q3 raw comments:", len(comments_q3))
print("Q4 raw comments:", len(comments_q4))

Q3 raw comments: 7443
Q4 raw comments: 8664
Q3 unique videos: 57
Q4 unique videos: 82


In [4]:
#remove duplicate comments for Q3
comments_q3 = comments_q3.drop_duplicates(
    subset=["video_id", "comment_author", "comment_text"]
)

#remove duplicate comments for Q4
comments_q4 = comments_q4.drop_duplicates(
    subset=["video_id", "comment_author", "comment_text"]
)

#show how many comments remain after removing duplicates
print("Q3 comments after duplicate removal:", len(comments_q3))
print("Q4 comments after duplicate removal:", len(comments_q4))

#save merged comment of Q3 into a CSV
comments_q3.to_csv(
    processed_folder / "Q3_comments_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

#save merged comment of Q4 into a CSV
comments_q4.to_csv(
    processed_folder / "Q4_comments_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

Q3 comments after duplicate removal: 4829
Q4 comments after duplicate removal: 6897
Saved:
data/processed/Q3_comments_raw.csv
data/processed/Q4_comments_raw.csv


In [5]:
# Define the clean function
def basic_clean(text):
    #convert the comments into a text format
    text = str(text)
    #convert HTML symbols into normal characters
    text = html.unescape(text)

    #remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    #remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

#clean Q3 and Q4 comments
comments_q3["comment_text_clean"] = comments_q3["comment_text"].apply(basic_clean)
comments_q4["comment_text_clean"] = comments_q4["comment_text"].apply(basic_clean)

#count words
comments_q3["word_count"] = comments_q3["comment_text_clean"].apply(lambda x: len(x.split()))
comments_q4["word_count"] = comments_q4["comment_text_clean"].apply(lambda x: len(x.split()))

#remove very short comments
clean_q3 = comments_q3[comments_q3["word_count"] >= 3].copy()
clean_q4 = comments_q4[comments_q4["word_count"] >= 3].copy()

#remove duplicate comments of Q3
clean_q3 = clean_q3.drop_duplicates(
    subset=["video_id", "comment_author", "comment_text_clean"]
)

#remove duplicate comments of Q3
clean_q4 = clean_q4.drop_duplicates(
    subset=["video_id", "comment_author", "comment_text_clean"]
)

#show how many comments remain after cleaning
print("Q3 comments after cleaning:", len(clean_q3))
print("Q4 comments after cleaning:", len(clean_q4))

Q3 comments after basic cleaning: 3587
Q4 comments after basic cleaning: 5550


In [6]:
#define language detection function 
def detect_language(text):
    #detect language from the comment text
    try:
        return detect(str(text))
    #if the language detection fails, it is marked as unknown 
    except LangDetectException:
        return "unknown"

#detect language using cleaned comment text
clean_q3["language"] = clean_q3["comment_text_clean"].apply(detect_language)
clean_q4["language"] = clean_q4["comment_text_clean"].apply(detect_language)

#keep English only
english_q3 = clean_q3[clean_q3["language"] == "en"].copy()
english_q4 = clean_q4[clean_q4["language"] == "en"].copy()

#show how many English comments remain
print("Q3 English comments:", len(english_q3))
print("Q4 English comments:", len(english_q4))

Q3 English comments: 2833
Q4 English comments: 4556


In [7]:
#download the NLTK stopword list
nltk.download("stopwords")

#load the english stopwords from NLTK
english_stopwords = set(stopwords.words("english"))

#extra stopwords that are common in the YouTube comments
custom_stopwords = {
    "video", "videos", "youtube", "like", "one",
    "would", "could", "really", "thing", "things", "make",
    "made", "get", "got", "see", "also", "even", "ai", "art",
    "people", "good", "think", "use", "using",
    "know", "time", "much", "say", "something",
    "want", "need", "first", "way", "better",
    "bro", "never", "still", "great", "polar", 
    "bears", "cake", "cat", "cats", "shrek", "maui",
    "skittles", "fruit", "baby", "lol", 
    "actually", "going", "looks", "im", "years", "look",
    "well", "every", "making", "dont", "put", "right",
    "said", "always", "thought", "thanks", "thank",
    "please", "someone", "take",
    "try", "lot", "many", "gummy", "chasing", "knife",
    "banana", "elmo", "walter",
    "white", "popeyes", "maui", "shrek", "lava",
    "lamp", "whisk", "ment", "mments", "oo", "op", 
    "blue", "red", "blaster", "co", "year", "old", 
    "scream", "screaming", "sun", "sunrise", "shoe",
    "run", "running", "sea", "ice", "hairy", "comfyui"
}

#combine the NLTK stopwords with the custom stopwords
all_stopwords = english_stopwords.union(custom_stopwords)

#define function
def tokenize_comment(text):
    text = str(text).lower()

    #keep only alphabetic words
    tokens = re.findall(r"[a-z]+", text)

    #remove single letter tokens, but keep "ai"
    tokens = [
        token for token in tokens
        if (len(token) > 1 or token == "ai")
        and token not in all_stopwords
    ]

    return tokens

#tokenise the Q3 comments
english_q3["tokens"] = english_q3["comment_text_clean"].apply(tokenize_comment)
english_q4["tokens"] = english_q4["comment_text_clean"].apply(tokenize_comment)

#tokenise the Q4 comments 
english_q3["tokens_joined"] = english_q3["tokens"].apply(lambda x: " ".join(x))
english_q4["tokens_joined"] = english_q4["tokens"].apply(lambda x: " ".join(x))

#remove comments where no useful tokens remain
final_q3 = english_q3[english_q3["tokens_joined"].str.len() > 0].copy()
final_q4 = english_q4[english_q4["tokens_joined"].str.len() > 0].copy()

#print number of comments after language filteing and token cleaning 
print("Q3 comments:", len(final_q3))
print("Q4 comments:", len(final_q4))

Q3 comments: 2768
Q4 comments: 4475


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\binh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
#save the cleaned and pre-processed Q3 comments into a CSV
final_q3.to_csv(
    processed_folder / "Q3_final_processed.csv",
    index=False,
    encoding="utf-8-sig"
)

#save the cleaned and pre-processed Q4 comments into a CSV
final_q4.to_csv(
    processed_folder / "Q4_final_processed.csv",
    index=False,
    encoding="utf-8-sig"
)

Saved:
Q3_final_processed.csv
Q4_final_processed.csv
